# Visual Genome Image Description Generation
## Notebook 00: Complete Setup and Data Download

This notebook handles everything needed to set up the project:
1. Environment setup and package installation
2. Data download from Stanford Visual Genome (direct)
3. Data preprocessing
4. Image download (optional)

**Run all cells in order for complete setup**

In [ ]:
# Install all required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers timm datasets tqdm omegaconf scikit-learn wandb
!pip install opencv-python pillow matplotlib seaborn plotly
!pip install jupyter ipykernel

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path
import json
import requests
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed

# Add src to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path}")

In [ ]:
# Test basic imports
try:
    import torch
    import torchvision
    import transformers
    import timm
    from tqdm import tqdm
    import omegaconf
    import sklearn
    print("✅ All core libraries imported successfully!")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please check package installation above")

In [ ]:
# Test project imports
try:
    from src.utils import load_config
    from src.data import run_preprocessing
    from src.features import FeatureExtractor
    print("✅ All project imports successful!")
except ImportError as e:
    print(f"❌ Project import error: {e}")
    print("Please check if src/ directory exists and __init__.py files are present")

In [ ]:
# Load configuration
config = load_config("../configs/config.yaml")
print("Configuration loaded:")
print(f"- Device: {config.device}")
print(f"- Data dir: {config.paths.data_dir}")
print(f"- Seed: {config.seed}")

In [ ]:
# Setup data directories
raw_dir = Path("../data/raw")
if not raw_dir.exists():
    print("Creating raw data directory...")
    raw_dir.mkdir(parents=True, exist_ok=True)

# Check for existing data files
expected_files = [
    "objects_v1.2.json",
    "attributes_v1.4.json",
    "relationships_v1.2.json"
]

existing_files = [f for f in expected_files if (raw_dir / f).exists()]
print(f"Found {len(existing_files)}/{len(expected_files)} data files")

if len(existing_files) < len(expected_files):
    print("Missing data files. Starting download from Stanford Visual Genome...")
    print("="*60)
    
    # Stanford Visual Genome download URLs (Version 1.2)
    DOWNLOAD_URLS = {
        "objects_v1.2.json": "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/objects_v1_2.json.zip",
        "attributes_v1.4.json": "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/attributes.json.zip",
        "relationships_v1.2.json": "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/relationships_v1_2.json.zip"
    }
    
    def download_and_extract_zip(url, extract_path, filename):
        """Download and extract a ZIP file from Stanford VG"""
        print(f"Downloading {filename} from Stanford Visual Genome...")
        
        # Download with progress indication
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        zip_path = extract_path / f"{filename}.zip"
        total_size = int(response.headers.get('content-length', 0))
        
        with open(zip_path, 'wb') as f:
            downloaded = 0
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    progress = int(50 * downloaded / total_size)
                    print(f"\rDownload progress: [{'=' * progress}{' ' * (50-progress)}] {downloaded/1024/1024:.1f}MB", end='', flush=True)
        print()  # New line after progress bar
        
        # Extract ZIP
        print(f"Extracting {filename}.zip...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Get the actual filename inside ZIP
            zip_files = zip_ref.namelist()
            json_file = next((f for f in zip_files if f.endswith('.json')), None)
            if json_file:
                zip_ref.extract(json_file, extract_path)
                # Rename to our expected filename
                extracted_path = extract_path / json_file
                target_path = extract_path / filename
                if extracted_path != target_path:
                    extracted_path.rename(target_path)
                print(f"✅ Extracted to {target_path}")
                
                # Show file size
                size_mb = target_path.stat().st_size / 1024 / 1024
                print(f"   File size: {size_mb:.1f} MB")
                return True
            else:
                print(f"❌ No JSON file found in {filename}.zip")
                return False
        
        # Clean up ZIP file
        zip_path.unlink()
        return True
    
    # Download each file
    for filename, url in DOWNLOAD_URLS.items():
        if (raw_dir / filename).exists():
            print(f"✅ {filename} already exists, skipping...")
            continue
            
        try:
            success = download_and_extract_zip(url, raw_dir, filename)
            if not success:
                print(f"❌ Failed to download {filename}")
                continue
        except Exception as e:
            print(f"❌ Error downloading {filename}: {e}")
            continue
    
    print("\n✅ Metadata download completed!")
    
    # Update existing files list
    existing_files = [f for f in expected_files if (raw_dir / f).exists()]
    print(f"After download: Found {len(existing_files)}/{len(expected_files)} data files")
    
else:
    print("✅ All data files present!")

In [ ]:
# Run preprocessing if data exists
if len(existing_files) == len(expected_files):
    print("Running data preprocessing...")
    try:
        run_preprocessing(
            raw_dir=str(raw_dir),
            processed_dir="../data/processed",
            max_objects=300,
            max_attributes=200,
            max_relations=100
        )
        print("✅ Preprocessing completed!")
    except Exception as e:
        print(f"❌ Preprocessing failed: {e}")
else:
    print("⚠️  Download data first before preprocessing")

In [ ]:
# Configuration for image download
IMAGE_DOWNLOAD_MODE = "sample"  # Options: "none", "sample", "all"

# "none": Skip image download
# "sample": Download 100 sample images for testing
# "all": Download all ~100K images (takes hours, ~20GB)

if IMAGE_DOWNLOAD_MODE == "sample" and len(existing_files) == len(expected_files):
    print("\nDownloading sample images for testing...")
    try:
        # Get sample image IDs
        objects_file = raw_dir / "objects_v1.2.json"
        with open(objects_file, 'r') as f:
            objects_data = json.load(f)
        
        # Get first 100 unique image IDs
        sample_image_ids = list(set(item['image_id'] for item in objects_data[:1000]))[:100]
        print(f"Selected {len(sample_image_ids)} sample images to download")
        
        # Download images function
        def download_vg_images(image_ids, image_dir, max_workers=4):
            """Download Visual Genome images from Stanford servers"""
            image_dir_path = Path(image_dir)
            image_dir_path.mkdir(parents=True, exist_ok=True)
            
            url_template = "https://cs.stanford.edu/people/rak248/VG_100K/{image_id}.jpg"
            results = {}
            
            # Check existing files
            for img_id in image_ids:
                local_path = image_dir_path / f"{img_id}.jpg"
                if local_path.exists():
                    results[img_id] = str(local_path)
            
            to_download = [img_id for img_id in image_ids if img_id not in results]
            print(f"Found {len(results)} existing images, downloading {len(to_download)} new ones")
            
            def _download_one(img_id):
                url = url_template.format(image_id=img_id)
                local_path = image_dir_path / f"{img_id}.jpg"
                try:
                    resp = requests.get(url, timeout=30)
                    resp.raise_for_status()
                    local_path.write_bytes(resp.content)
                    return img_id, str(local_path)
                except Exception as e:
                    print(f"Failed to download image {img_id}: {e}")
                    return img_id, None
            
            if to_download:
                with ThreadPoolExecutor(max_workers=max_workers) as executor:
                    futures = {executor.submit(_download_one, img_id): img_id for img_id in to_download}
                    for future in tqdm(as_completed(futures), total=len(to_download), desc="Downloading images"):
                        img_id, path = future.result()
                        if path:
                            results[img_id] = path
                
                success = sum(1 for v in results.values() if v and Path(v).exists())
                print(f"✅ Downloaded {success}/{len(sample_image_ids)} sample images")
            else:
                print("✅ All sample images already exist")
        
        # Download sample images
        download_vg_images(sample_image_ids, str(raw_dir / "images"), max_workers=4)
        
    except Exception as e:
        print(f"❌ Sample image download failed: {e}")
        print("You can continue with training using on-demand image loading")
        
elif IMAGE_DOWNLOAD_MODE == "all" and len(existing_files) == len(expected_files):
    print("\n" + "="*60)
    print("DOWNLOADING ALL IMAGES (This will take hours/days!)")
    print("="*60)
    
    try:
        # Get all image IDs from objects dataset
        objects_file = raw_dir / "objects_v1.2.json"
        with open(objects_file, 'r') as f:
            objects_data = json.load(f)
        
        all_image_ids = list(set(item['image_id'] for item in objects_data))
        print(f"Found {len(all_image_ids)} unique images to download")
        
        # Download in batches to avoid memory issues
        batch_size = 1000
        for i in range(0, len(all_image_ids), batch_size):
            batch_ids = all_image_ids[i:i+batch_size]
            print(f"Downloading batch {i//batch_size + 1}/{(len(all_image_ids)-1)//batch_size + 1} ({len(batch_ids)} images)")
            
            download_vg_images(batch_ids, str(raw_dir / "images"), max_workers=8)
        
        print("✅ All images downloaded!")
        
    except Exception as e:
        print(f"❌ Full image download failed: {e}")
        print("Consider downloading images on-demand during training")
        
else:
    print("\nSkipping image download (set IMAGE_DOWNLOAD_MODE to 'sample' or 'all' to download images)")

print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)
print("Next steps:")
print("1. Run notebook 01_data_exploration.ipynb to explore the data")
print("2. Run notebook 02_feature_extraction.ipynb for feature caching")
print("3. Run task1_train.ipynb and task2_train.ipynb for training")

# Visual Genome Image Description Generation
## Notebook 00: Complete Setup and Data Download

This notebook handles everything needed to set up the project:
1. Environment setup and package installation
2. Data download from HuggingFace
3. Data preprocessing
4. Image download (optional)

**Run all cells in order for complete setup**

In [20]:
# Install all required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers timm datasets tqdm omegaconf scikit-learn wandb
!pip install opencv-python pillow matplotlib seaborn plotly
!pip install jupyter ipykernel

Looking in indexes: https://download.pytorch.org/whl/cu118


In [21]:
# Import required libraries
import os
import sys
from pathlib import Path
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# Add src to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path}")

Project root: c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj
Python path: ['c:\\Users\\caoha\\OneDrive\\Desktop\\study\\Ky2\\CV\\prj', 'c:\\Users\\caoha\\anaconda3\\python313.zip', 'c:\\Users\\caoha\\anaconda3\\DLLs', 'c:\\Users\\caoha\\anaconda3\\Lib', 'c:\\Users\\caoha\\anaconda3', '', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages\\win32', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages\\Pythonwin']


In [22]:
# Test basic imports
try:
    import torch
    import torchvision
    import transformers
    import timm
    from datasets import load_dataset
    from tqdm import tqdm
    import omegaconf
    import sklearn
    print("✅ All core libraries imported successfully!")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please check package installation above")

✅ All core libraries imported successfully!
PyTorch version: 2.10.0+cpu
CUDA available: False


In [23]:
# Test project imports
try:
    from src.utils import load_config
    from src.data import run_preprocessing
    from src.features import FeatureExtractor
    print("✅ All project imports successful!")
except ImportError as e:
    print(f"❌ Project import error: {e}")
    print("Please check if src/ directory exists and __init__.py files are present")

✅ All project imports successful!


In [24]:
# Load configuration
config = load_config("../configs/config.yaml")
print("Configuration loaded:")
print(f"- Device: {config.device}")
print(f"- Data dir: {config.paths.data_dir}")
print(f"- Seed: {config.seed}")

Configuration loaded:
- Device: cuda
- Data dir: data
- Seed: 42


In [25]:
# Setup data directories
raw_dir = Path("../data/raw")
if not raw_dir.exists():
    print("Creating raw data directory...")
    raw_dir.mkdir(parents=True, exist_ok=True)

# Check for existing data files
expected_files = [
    "objects_v1.2.json",
    "attributes_v1.4.json",
    "relationships_v1.2.json"
]

existing_files = [f for f in expected_files if (raw_dir / f).exists()]
print(f"Found {len(existing_files)}/{len(expected_files)} data files")

if len(existing_files) < len(expected_files):
    print("Missing data files. Starting download from HuggingFace...")
    print("="*60)
    
    # Download configuration
    DATASET_REPO_ID = "ranjaykrishna/visual_genome"
    CACHE_DIR = raw_dir / "hf_cache"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    
    def download_vg_subset(subset: str, cache_dir: str, split: str = "train"):
        """Download a subset from HuggingFace Visual Genome dataset"""
        print(f"Downloading subset '{subset}' from {DATASET_REPO_ID} ...")
        dataset = load_dataset(
            DATASET_REPO_ID,
            subset,
            split=split,
            cache_dir=cache_dir,
            trust_remote_code=True,
        )
        print(f"✅ Downloaded: {len(dataset)} samples")
        return dataset
    
    def save_dataset_as_json(dataset, save_path: str):
        """Save HuggingFace dataset to JSON file"""
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        
        print(f"Saving {len(dataset)} records to {save_path}")
        data = [dict(item) for item in tqdm(dataset)]
        
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        size_mb = save_path.stat().st_size / 1024 / 1024
        print(f"✅ Saved: {save_path} ({size_mb:.1f} MB)")
    
    # Download all metadata subsets
    subsets_to_download = [
        ("objects_v1.2.0", "objects_v1.2.json"),
        ("attributes_v1.4.0", "attributes_v1.4.json"),
        ("relationships_v1.2.0", "relationships_v1.2.json"),
    ]
    
    for subset_name, filename in subsets_to_download:
        try:
            print(f"\n--- Downloading {subset_name} ---")
            dataset = download_vg_subset(subset_name, str(CACHE_DIR))
            output_path = raw_dir / filename
            save_dataset_as_json(dataset, str(output_path))
        except Exception as e:
            print(f"❌ Failed to download {subset_name}: {e}")
            continue
    
    print("\n✅ Metadata download completed!")
    
    # Update existing files list
    existing_files = [f for f in expected_files if (raw_dir / f).exists()]
    print(f"After download: Found {len(existing_files)}/{len(expected_files)} data files")
    
else:
    print("✅ All data files present!")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ranjaykrishna/visual_genome' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Found 0/3 data files
Missing data files. Starting download from HuggingFace...

--- Downloading objects_v1.2.0 ---


README.md: 0.00B [00:00, ?B/s]

c:\Users\caoha\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\caoha\.cache\huggingface\hub\datasets--ranjaykrishna--visual_genome. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


visual_genome.py: 0.00B [00:00, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ranjaykrishna/visual_genome' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


❌ Failed to download objects_v1.2.0: Dataset scripts are no longer supported, but found visual_genome.py

--- Downloading attributes_v1.4.0 ---


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ranjaykrishna/visual_genome' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


❌ Failed to download attributes_v1.4.0: Dataset scripts are no longer supported, but found visual_genome.py

--- Downloading relationships_v1.2.0 ---
❌ Failed to download relationships_v1.2.0: Dataset scripts are no longer supported, but found visual_genome.py

✅ Metadata download completed!
After download: Found 0/3 data files


In [26]:
# Run preprocessing if data exists
if len(existing_files) == len(expected_files):
    print("Running data preprocessing...")
    try:
        run_preprocessing(
            raw_dir=str(raw_dir),
            processed_dir="../data/processed",
            max_objects=300,
            max_attributes=200,
            max_relations=100
        )
        print("✅ Preprocessing completed!")
    except Exception as e:
        print(f"❌ Preprocessing failed: {e}")
else:
    print("⚠️  Download data first before preprocessing")

⚠️  Download data first before preprocessing


In [27]:
# Configuration for image download
IMAGE_DOWNLOAD_MODE = "sample"  # Options: "none", "sample", "all"

# "none": Skip image download
# "sample": Download 100 sample images for testing
# "all": Download all ~100K images (takes hours, ~20GB)

if IMAGE_DOWNLOAD_MODE == "sample" and len(existing_files) == len(expected_files):
    print("\nDownloading sample images for testing...")
    try:
        # Get sample image IDs
        objects_file = raw_dir / "objects_v1.2.json"
        with open(objects_file, 'r') as f:
            objects_data = json.load(f)
        
        # Get first 100 unique image IDs
        sample_image_ids = list(set(item['image_id'] for item in objects_data[:1000]))[:100]
        print(f"Selected {len(sample_image_ids)} sample images to download")
        
        # Download images function
        def download_vg_images(image_ids, image_dir, max_workers=4):
            """Download Visual Genome images from Stanford servers"""
            image_dir_path = Path(image_dir)
            image_dir_path.mkdir(parents=True, exist_ok=True)
            
            url_template = "https://cs.stanford.edu/people/rak248/VG_100K/{image_id}.jpg"
            results = {}
            
            # Check existing files
            for img_id in image_ids:
                local_path = image_dir_path / f"{img_id}.jpg"
                if local_path.exists():
                    results[img_id] = str(local_path)
            
            to_download = [img_id for img_id in image_ids if img_id not in results]
            print(f"Found {len(results)} existing images, downloading {len(to_download)} new ones")
            
            def _download_one(img_id):
                url = url_template.format(image_id=img_id)
                local_path = image_dir_path / f"{img_id}.jpg"
                try:
                    resp = requests.get(url, timeout=30)
                    resp.raise_for_status()
                    local_path.write_bytes(resp.content)
                    return img_id, str(local_path)
                except Exception as e:
                    print(f"Failed to download image {img_id}: {e}")
                    return img_id, None
            
            if to_download:
                with ThreadPoolExecutor(max_workers=max_workers) as executor:
                    futures = {executor.submit(_download_one, img_id): img_id for img_id in to_download}
                    for future in tqdm(as_completed(futures), total=len(to_download), desc="Downloading images"):
                        img_id, path = future.result()
                        if path:
                            results[img_id] = path
                
                success = sum(1 for v in results.values() if v and Path(v).exists())
                print(f"✅ Downloaded {success}/{len(sample_image_ids)} sample images")
            else:
                print("✅ All sample images already exist")
        
        # Download sample images
        download_vg_images(sample_image_ids, str(raw_dir / "images"), max_workers=4)
        
    except Exception as e:
        print(f"❌ Sample image download failed: {e}")
        print("You can continue with training using on-demand image loading")
        
elif IMAGE_DOWNLOAD_MODE == "all" and len(existing_files) == len(expected_files):
    print("\n" + "="*60)
    print("DOWNLOADING ALL IMAGES (This will take hours/days!)")
    print("="*60)
    
    try:
        # Get all image IDs from objects dataset
        objects_file = raw_dir / "objects_v1.2.json"
        with open(objects_file, 'r') as f:
            objects_data = json.load(f)
        
        all_image_ids = list(set(item['image_id'] for item in objects_data))
        print(f"Found {len(all_image_ids)} unique images to download")
        
        # Download in batches to avoid memory issues
        batch_size = 1000
        for i in range(0, len(all_image_ids), batch_size):
            batch_ids = all_image_ids[i:i+batch_size]
            print(f"Downloading batch {i//batch_size + 1}/{(len(all_image_ids)-1)//batch_size + 1} ({len(batch_ids)} images)")
            
            download_vg_images(batch_ids, str(raw_dir / "images"), max_workers=8)
        
        print("✅ All images downloaded!")
        
    except Exception as e:
        print(f"❌ Full image download failed: {e}")
        print("Consider downloading images on-demand during training")
        
else:
    print("\nSkipping image download (set IMAGE_DOWNLOAD_MODE to 'sample' or 'all' to download images)")

print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)
print("Next steps:")
print("1. Run notebook 01_data_exploration.ipynb to explore the data")
print("2. Run notebook 02_feature_extraction.ipynb for feature caching")
print("3. Run task1_train.ipynb and task2_train.ipynb for training")


Skipping image download (set IMAGE_DOWNLOAD_MODE to 'sample' or 'all' to download images)

SETUP COMPLETE
Next steps:
1. Run notebook 01_data_exploration.ipynb to explore the data
2. Run notebook 02_feature_extraction.ipynb for feature caching
3. Run task1_train.ipynb and task2_train.ipynb for training


# Visual Genome Image Description Generation
## Notebook 00: Setup and Data Download

This notebook handles:
1. Environment setup and package installation
2. Data download from HuggingFace
3. Basic verification of downloaded data

In [28]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path}")

Project root: c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj
Python path: ['c:\\Users\\caoha\\OneDrive\\Desktop\\study\\Ky2\\CV\\prj', 'c:\\Users\\caoha\\anaconda3\\python313.zip', 'c:\\Users\\caoha\\anaconda3\\DLLs', 'c:\\Users\\caoha\\anaconda3\\Lib', 'c:\\Users\\caoha\\anaconda3', '', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages\\win32', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\caoha\\anaconda3\\Lib\\site-packages\\Pythonwin']


In [29]:
# Test imports
try:
    from src.utils import load_config
    from src.data import download_vg_subset, run_preprocessing
    from src.features import FeatureExtractor
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please run: pip install -r requirements.txt")

✅ All imports successful!


In [30]:
# Load configuration
config = load_config("../configs/config.yaml")
print("Configuration loaded:")
print(f"- Device: {config.device}")
print(f"- Data dir: {config.paths.data_dir}")
print(f"- Seed: {config.seed}")

Configuration loaded:
- Device: cuda
- Data dir: data
- Seed: 42


In [31]:
# Check if data exists
raw_dir = Path("../data/raw")
if not raw_dir.exists():
    print("Creating raw data directory...")
    raw_dir.mkdir(parents=True, exist_ok=True)

# Check for existing data files
expected_files = [
    "objects_v1.2.json",
    "attributes_v1.4.json",
    "relationships_v1.2.json"
]

existing_files = [f for f in expected_files if (raw_dir / f).exists()]
print(f"Found {len(existing_files)}/{len(expected_files)} data files")

if len(existing_files) < len(expected_files):
    print("Missing data files. Starting download from HuggingFace...")
    print("="*60)

    # Import required libraries for download
    try:
        from datasets import load_dataset
        from tqdm import tqdm
        import json
        print("✅ Download libraries imported successfully!")
    except ImportError as e:
        print(f"❌ Missing libraries: {e}")
        print("Please install: pip install datasets tqdm")
        raise

    # Download configuration
    DATASET_REPO_ID = "ranjaykrishna/visual_genome"
    CACHE_DIR = raw_dir / "hf_cache"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    def download_vg_subset(subset: str, cache_dir: str, split: str = "train"):
        """Download a subset from HuggingFace Visual Genome dataset"""
        print(f"Downloading subset '{subset}' from {DATASET_REPO_ID} ...")
        dataset = load_dataset(
            DATASET_REPO_ID,
            subset,
            split=split,
            cache_dir=cache_dir,
            trust_remote_code=True,
        )
        print(f"✅ Downloaded: {len(dataset)} samples")
        return dataset

    def save_dataset_as_json(dataset, save_path: str):
        """Save HuggingFace dataset to JSON file"""
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"Saving {len(dataset)} records to {save_path}")
        data = [dict(item) for item in tqdm(dataset)]

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        size_mb = save_path.stat().st_size / 1024 / 1024
        print(f"✅ Saved: {save_path} ({size_mb:.1f} MB)")

    # Download all metadata subsets
    subsets_to_download = [
        ("objects_v1.2.0", "objects_v1.2.json"),
        ("attributes_v1.4.0", "attributes_v1.4.json"),
        ("relationships_v1.2.0", "relationships_v1.2.json"),
    ]

    for subset_name, filename in subsets_to_download:
        try:
            print(f"\n--- Downloading {subset_name} ---")
            dataset = download_vg_subset(subset_name, str(CACHE_DIR))
            output_path = raw_dir / filename
            save_dataset_as_json(dataset, str(output_path))
        except Exception as e:
            print(f"❌ Failed to download {subset_name}: {e}")
            continue

    print("\n✅ Metadata download completed!")

    # Update existing files list
    existing_files = [f for f in expected_files if (raw_dir / f).exists()]
    print(f"After download: Found {len(existing_files)}/{len(expected_files)} data files")

    except Exception as e:
        print(f"❌ Download failed: {e}")
        print("Please check your internet connection and try again")
        print("You can also download manually from HuggingFace")

else:
    print("✅ All data files present!")

SyntaxError: invalid syntax (2580341360.py, line 87)

In [ ]:
# Run preprocessing if data exists
if len(existing_files) == len(expected_files):
    print("Running data preprocessing...")
    try:
        run_preprocessing(
            raw_dir=str(raw_dir),
            processed_dir="../data/processed",
            max_objects=300,
            max_attributes=200,
            max_relations=100
        )
        print("✅ Preprocessing completed!")
    except Exception as e:
        print(f"❌ Preprocessing failed: {e}")
else:
    print("⚠️  Download data first before preprocessing")

print("\nNotebook setup complete!")

⚠️  Download data first before preprocessing

Notebook setup complete!


In [ ]:
# Configuration for image download
IMAGE_DOWNLOAD_MODE = "sample"  # Options: "none", "sample", "all"

# "none": Skip image download
# "sample": Download 100 sample images for testing
# "all": Download all ~100K images (takes hours, ~20GB)

if IMAGE_DOWNLOAD_MODE == "sample" and len(existing_files) == len(expected_files):
    print("\nDownloading sample images for testing...")
    try:
        import requests
        from concurrent.futures import ThreadPoolExecutor, as_completed
        
        # Get sample image IDs
        objects_file = raw_dir / "objects_v1.2.json"
        with open(objects_file, 'r') as f:
            objects_data = json.load(f)

        # Get first 100 unique image IDs
        sample_image_ids = list(set(item['image_id'] for item in objects_data[:1000]))[:100]
        print(f"Selected {len(sample_image_ids)} sample images to download")

        # Download images function
        def download_vg_images(image_ids, image_dir, max_workers=4):
            """Download Visual Genome images from Stanford servers"""
            image_dir_path = Path(image_dir)
            image_dir_path.mkdir(parents=True, exist_ok=True)
            
            url_template = "https://cs.stanford.edu/people/rak248/VG_100K/{image_id}.jpg"
            results = {}
            
            # Check existing files
            for img_id in image_ids:
                local_path = image_dir_path / f"{img_id}.jpg"
                if local_path.exists():
                    results[img_id] = str(local_path)
            
            to_download = [img_id for img_id in image_ids if img_id not in results]
            print(f"Found {len(results)} existing images, downloading {len(to_download)} new ones")
            
            def _download_one(img_id):
                url = url_template.format(image_id=img_id)
                local_path = image_dir_path / f"{img_id}.jpg"
                try:
                    resp = requests.get(url, timeout=30)
                    resp.raise_for_status()
                    local_path.write_bytes(resp.content)
                    return img_id, str(local_path)
                except Exception as e:
                    print(f"Failed to download image {img_id}: {e}")
                    return img_id, None
            
            if to_download:
                with ThreadPoolExecutor(max_workers=max_workers) as executor:
                    futures = {executor.submit(_download_one, img_id): img_id for img_id in to_download}
                    for future in tqdm(as_completed(futures), total=len(to_download), desc="Downloading images"):
                        img_id, path = future.result()
                        if path:
                            results[img_id] = path
                
                success = sum(1 for v in results.values() if v and Path(v).exists())
                print(f"✅ Downloaded {success}/{len(sample_image_ids)} sample images")
            else:
                print("✅ All sample images already exist")

        # Download sample images
        download_vg_images(sample_image_ids, str(raw_dir / "images"), max_workers=4)

    except Exception as e:
        print(f"❌ Sample image download failed: {e}")
        print("You can continue with training using on-demand image loading")

elif IMAGE_DOWNLOAD_MODE == "all" and len(existing_files) == len(expected_files):
    print("\n" + "="*60)
    print("DOWNLOADING ALL IMAGES (This will take hours/days!)")
    print("="*60)
    
    try:
        import requests
        from concurrent.futures import ThreadPoolExecutor, as_completed
        
        # Get all image IDs from objects dataset
        objects_file = raw_dir / "objects_v1.2.json"
        with open(objects_file, 'r') as f:
            objects_data = json.load(f)

        all_image_ids = list(set(item['image_id'] for item in objects_data))
        print(f"Found {len(all_image_ids)} unique images to download")

        # Download in batches to avoid memory issues
        batch_size = 1000
        for i in range(0, len(all_image_ids), batch_size):
            batch_ids = all_image_ids[i:i+batch_size]
            print(f"Downloading batch {i//batch_size + 1}/{(len(all_image_ids)-1)//batch_size + 1} ({len(batch_ids)} images)")

            download_vg_images(batch_ids, str(raw_dir / "images"), max_workers=8)

        print("✅ All images downloaded!")

    except Exception as e:
        print(f"❌ Full image download failed: {e}")
        print("Consider downloading images on-demand during training")

else:
    print("\nSkipping image download (set IMAGE_DOWNLOAD_MODE to 'sample' or 'all' to download images)")

print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)
print("Next steps:")
print("1. Run this notebook again to verify data is downloaded")
print("2. Run notebook 02_data_processing.ipynb for preprocessing")
print("3. Run notebook 03_feature_extraction.ipynb for feature caching")
print("4. Run task1_train.ipynb and task2_train.ipynb for training")


Skipping full image download (set DOWNLOAD_ALL_IMAGES=True to download all)

SETUP COMPLETE
Next steps:
1. Run this notebook again to verify data is downloaded
2. Run notebook 02_data_processing.ipynb for preprocessing
3. Run notebook 03_feature_extraction.ipynb for feature caching
4. Run task1_train.ipynb and task2_train.ipynb for training
